# 📊 NeMo Fraud Detection: LoRA Adapter Evaluation

This notebook performs a local evaluation of the fine-tuned LoRA adapter (**Llama-3.1-8B-Instruct**) for fraud detection, featuring full **End-to-End Tracking with Weights & Biases (W&B)**.

### Features:
1. **Model Initialization:** Loads the base model and attaches the trained LoRA adapter.
2. **Inference:** Generates responses locally via PyTorch/Transformers using validation transcripts.
3. **Parsing & Validation:** Maps ground truth labels and robustly compares them with model outputs.
4. **Metrics & W&B Tracking:** Computes overall accuracy, logs detailed evaluation tables, metrics, and report artifacts to Weights & Biases (`wandb`).

In [ ]:
import json
import torch
import pandas as pd
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import wandb

# Configuration of paths and models
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
ADAPTER_PATH = "/data/nemo-fraud-detection-notebooks/notebooks/04_FineTuning/evaluation/fraud_detection_qlora"
VAL_FILE = "/data/nemo-fraud-detection-notebooks/notebooks/02_Data_Curation/data/sft/validation.jsonl"

print("✅ Libraries and configurations loaded successfully.")

### 1. Execution of LoRA Adapter Evaluation with W&B Tracking

In [ ]:
def run_lora_evaluation():
    print("🚀 Loading base model and LoRA adapter...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, 
        torch_dtype=torch.float16, 
        device_map="auto"
    )
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    model.eval()

    # Initialize Weights & Biases run with comprehensive config
    wandb.init(
        project="nemo-fraud-detection", 
        name="lora-evaluation",
        config={
            "eval_pipeline": "lora-adapter-evaluation",
            "base_model": BASE_MODEL,
            "adapter_path": ADAPTER_PATH,
            "validation_file": VAL_FILE,
            "max_new_tokens": 5
        }
    )
    
    correct = 0
    total = 0
    results_list = []
    
    # Create a W&B Table for detailed sample-level tracking
    eval_table = wandb.Table(columns=["sample_index", "transcript", "true_label", "model_raw_response", "model_parsed_answer", "is_correct"])

    print("🚀 Starting local LoRA inference and evaluation...\n")
    with open(VAL_FILE, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            input_content = data.get("input", "")
            raw_true = data.get("output", "").strip().lower()
            
            if not input_content:
                continue
                
            # Format input using SFT prompt structure
            input_text = f"### Instruction:\n{input_content}\n\n### Response:\n"
            
            inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, 
                    max_new_tokens=5,
                    temperature=0.0,
                    do_sample=False
                )
            
            response = tokenizer.decode(outputs[0], skip_special_tokens=True).split("### Response:\n")[-1].strip().lower()
            
            # Standardize ground truth and model predictions
            true_label = "fraud" if ("betrug" in raw_true or "fraud" in raw_true) and "kein" not in raw_true else "legitimate"
            model_answer = "fraud" if "fraud" in response and "legitimate" not in response else ("legitimate" if "legitimate" in response else response)
            
            is_correct = (true_label == model_answer)
            if is_correct:
                correct += 1
            total += 1
            
            # Add data to W&B Table
            eval_table.add_data(total, input_content[:200] + "...", true_label, response, model_answer, is_correct)
            results_list.append({
                "id": total,
                "true_label": true_label,
                "model_prediction": model_answer,
                "model_raw_response": response,
                "is_correct": is_correct
            })
            
            print(f"Sample {total} | Expected: {true_label} | Predicted: {model_answer} | Correct: {is_correct}")

    accuracy = (correct / total) * 100 if total > 0 else 0
    
    print("\n" + "="*45)
    print("📊 LORA EVALUATION RESULTS")
    print("="*45)
    print(f"✅ Correctly identified: {correct} / {total}")
    print(f"🎯 Accuracy:             {accuracy:.2f}%")
    print("="*45)

    # Save evaluation report as CSV locally
    df = pd.DataFrame(results_list)
    os.makedirs("/data/nemo-fraud-detection/notebooks/03_Evaluation/data", exist_ok=True)
    csv_path = "/data/nemo-fraud-detection/notebooks/03_Evaluation/data/lora_evaluation_report.csv"
    df.to_csv(csv_path, index=False)

    # Log aggregate metrics, interactive table, and artifacts to W&B
    wandb.log({
        "eval/accuracy": accuracy,
        "eval/correct_predictions": correct,
        "eval/total_samples": total,
        "eval/predictions_table": eval_table
    })

    # Version CSV report as an Artifact
    artifact = wandb.Artifact(
        name="lora-evaluation-report",
        type="evaluation-report",
        description="Detailed LoRA fine-tuned model evaluation results for fraud detection"
    )
    artifact.add_file(csv_path)
    wandb.log_artifact(artifact)

    wandb.finish()
    print("✅ W&B LoRA evaluation logging completed successfully!")

# Start LoRA Evaluation
run_lora_evaluation()